<pre>
- Ozônio
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.


In [9]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [10]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [11]:
dataset = "cams-global-reanalysis-eac4"
request = {
    "variable": [
        "ozone"
    ],
    "pressure_level": ["1000"],
    "date": ["2025-12-01/2025-12-31"],
    "time": ["06:00"],
    "data_format": "netcdf",
    "area": [6, -74, -35, -34]
}

client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

ret_download = client.retrieve(dataset, request).download()

print(f"Download completed: {ret_download}")

2026-07-22 15:57:47,006 INFO Request ID is 1ac68552-5607-4cad-87b1-24b3c82b1c8b
2026-07-22 15:57:47,197 INFO status has been updated to accepted
2026-07-22 15:58:01,309 INFO status has been updated to running
2026-07-22 15:58:09,106 INFO status has been updated to successful
                                                                                      

Download completed: 5eaba1ac52ea432e4eec14e4882ef4d6.nc


In [12]:
with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Poluicao\\{ret_download}"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:
    # Transforma o Dataset em um Spark Dataframe
    df_dask   = ds.to_dask_dataframe()
    df_dask_c = df_dask.compute()
    df_ozonio = spark.createDataFrame(df_dask_c)

df_ozonio.printSchema()
df_ozonio.show(10, False)

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- valid_time: timestamp (nullable = true)
 |-- pressure_level: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- go3: float (nullable = true)

+-------------------+--------------+--------+---------+------------+
|valid_time         |pressure_level|latitude|longitude|go3         |
+-------------------+--------------+--------+---------+------------+
|2025-12-01 06:00:00|1000.0        |6.0     |-73.5    |7.943849E-9 |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.75   |6.079655E-9 |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.0    |1.1371436E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-71.25   |2.0574259E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-70.5    |2.804438E-8 |
|2025-12-01 06:00:00|1000.0        |6.0     |-69.75   |2.8920834E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-69.0    |2.1468974E-8|
|2025-12-01 06:00:00|1000.0        |6.0     |-68.25   |1.7145506E-8|
|2025-12-01 06:00:00|1000.0  

In [13]:
# Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)

M_AR = 28.9644 # g/mol
M_O3 = 47.9982 # g/mol
FATOR_CONVERSAO = (M_AR / M_O3) * 1e9  # ~ 1.03407e9

drop_cols = ["valid_time", "pressure_level", "go3"]

df_ozonio_ppb = \
    (df_ozonio
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("Poluição do ar - O₃ (ppb)") 
                     ,"valor": (F.col("go3") * F.lit(FATOR_CONVERSAO)).cast("double")
                     ,"unidade_medida": F.lit("ppb")})
         .drop(*drop_cols)

    )

In [14]:
# df_ozonio_ppb.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_ozonio.csv", index=False)

df_ozonio_ppb.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_ozonio.parquet")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [15]:
df_ozonio_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_ozonio.parquet")

df_ozonio_parquet.printSchema()
df_ozonio_parquet.show(10, False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+--------+---------+------------+-------------------------+------------------+--------------+
|latitude|longitude|data_medicao|indicador                |valor             |unidade_medida|
+--------+---------+------------+-------------------------+------------------+--------------+
|6.0     |-73.5    |2025-12-01  |Poluição do ar - O₃ (ppb)|4.793696927349565 |ppb           |
|6.0     |-72.75   |2025-12-01  |Poluição do ar - O₃ (ppb)|3.6687533976267988|ppb           |
|6.0     |-72.0    |2025-12-01  |Poluição do ar - O₃ (ppb)|6.862066238846523 |ppb           |
|6.0     |-71.25   |2025-12-01  |Poluição do ar - O₃ (ppb)|12.415487661656732|ppb           |
|6.0     |-70.5    |2025-12-01  |Poluição do ar - O₃ (ppb)|16.923314181233945|ppb        

In [16]:
os.remove(r"C:\Marco Conti\Projetos\MAIS-v2\Poluicao\{file_name}".format(file_name = ret_download))